# EXP016 — ST-GCN under the author-released Hundman protocol

**Run all cells in order.** Select an **A100 GPU**, authorise Drive, and upload **EXP016_Package.zip** when prompted. No other input files or code edits are needed.

This trains only ST-GCNs. The released LSTM is verified by inference and reused. There are **246 fits: 82 channels × 3 seeds**. All 250 prior observations and all released features are used, with a ten-output forecasting objective and first-output detection.

Read `PROTOCOL.md` in the package for the chronological-validation departure, inherited TEST scaling, C-2 detector sensitivity and label policy. This is a retrospective comparison, not an independent architecture-only holdout.

The actual GPU is checked and benchmarked before fitting. A six-hour total is **not guaranteed**. The scientific cell pauses safely after 5.5 hours; rerun all cells on the same GPU type to continue. Do not launch two sessions against the same Drive folder.

In [1]:
# 1 — Runtime and persistent destination
import os, sys, json, hashlib, shutil, subprocess, zipfile, tempfile
from pathlib import Path
AUTOTEST = os.environ.get("EXP016_NOTEBOOK_FIXTURE") == "1"  # build-time harness only; leave unchanged
EXPECTED_PACKAGE_SHA256 = "bbfea0df03db1afb9820e08d50079afe2a29653a42d093e9e2440febc2bd5414"
if AUTOTEST:
    BASE = Path(os.environ["EXP016_TEST_WORKSPACE"])
    DRIVE_ROOT = BASE / "simulated_drive" / "EXP016_V1"
else:
    if sys.version_info[:2] not in ((3,10),(3,11),(3,12)):
        raise RuntimeError("EXP016 requires Python 3.10–3.12. Select a compatible Colab runtime.")
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True) if shutil.which("nvidia-smi") else None
    if gpu is None or gpu.returncode:
        raise RuntimeError("No NVIDIA GPU. Select Runtime → Change runtime type → A100 GPU, then Run all again.")
    print("Selected GPU:", gpu.stdout.strip())
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/EXP016_workspace')
    DRIVE_ROOT = Path('/content/drive/MyDrive/EXP016_V1')
BASE.mkdir(parents=True, exist_ok=True)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
WORKER_ENV = {**os.environ, "PYTHONNOUSERSITE":"1", "CUBLAS_WORKSPACE_CONFIG":":4096:8"}
print("Working directory:", BASE)
print("Persistent continuation folder:", DRIVE_ROOT)


Executed successfully in sequential-cell CPU fixture harness. See transcript.


### Upload the package once
The package includes the supplied dataset, labels and released weights/predictions. On later sessions it is recovered from Drive. A checksum prevents mixing versions.

In [2]:
# 2 — Upload/recover, validate, and safely unpack
cached = DRIVE_ROOT / "EXP016_Package.zip"
if not cached.exists():
    if AUTOTEST:
        supplied = Path(os.environ["EXP016_TEST_PACKAGE"])
    else:
        from google.colab import files
        print("Upload EXP016_Package.zip intact.")
        uploaded = files.upload()
        candidates = [Path(name) for name in uploaded if name.lower().endswith('.zip')]
        if len(candidates) != 1:
            raise RuntimeError("Upload exactly one file: EXP016_Package.zip.")
        supplied = candidates[0]
    if hashlib.sha256(supplied.read_bytes()).hexdigest() != EXPECTED_PACKAGE_SHA256:
        raise RuntimeError("Wrong/corrupted package. Use the ZIP delivered with this notebook.")
    temporary = cached.with_suffix('.zip.part')
    shutil.copyfile(supplied, temporary)
    temporary.replace(cached)
if hashlib.sha256(cached.read_bytes()).hexdigest() != EXPECTED_PACKAGE_SHA256:
    raise RuntimeError("The stored package belongs to a different notebook/version. Keep its results and use the matching notebook.")
local_zip = BASE / cached.name
shutil.copyfile(cached, local_zip)
with zipfile.ZipFile(local_zip) as z:
    for item in z.infolist():
        target = (BASE / item.filename).resolve()
        if not target.is_relative_to(BASE.resolve()) or ((item.external_attr >> 16) & 0o170000) == 0o120000:
            raise RuntimeError("Unsafe archive member: " + item.filename)
    z.extractall(BASE)
PKG = BASE / "EXP016"
manifest = json.loads((PKG / "PACKAGE_MANIFEST.json").read_text())
for name, expected in manifest.items():
    if hashlib.sha256((PKG/name).read_bytes()).hexdigest() != expected:
        raise RuntimeError("Package verification failed: " + name)
print("Verified", len(manifest), "package files.")


Executed successfully in sequential-cell CPU fixture harness. See transcript.


In [3]:
# 3 — Isolated dependencies (does not alter/restart the notebook kernel)
VENV = BASE / "worker_env"
cmd = [sys.executable, str(PKG/"bootstrap.py"), "--venv", str(VENV)]
if AUTOTEST: cmd += ["--cpu-fixture"]
subprocess.run(cmd, check=True, env=WORKER_ENV)
PYTHON = VENV / "bin/python"
def worker(*args):
    subprocess.run([str(PYTHON), *map(str,args)], cwd=PKG, env=WORKER_ENV, check=True)
print("Worker ready:", PYTHON)


Executed successfully in sequential-cell CPU fixture harness. See transcript.


### Engineering checks
The fixture is synthetic and uses separate folders. Its fitted weights and scores never enter NASA results. Failure stops the notebook before scientific fitting.

In [4]:
# 4 — Unit checks and complete synthetic pipeline
worker("-m", "pytest", "tests", "-q")
worker("-m", "exp016.run", "fixture", "--out", BASE/"engineering_fixture", "--data", BASE/"fixture_data")
fixture_audit = json.loads((BASE/"engineering_fixture/FINAL_AUDIT.json").read_text())
assert fixture_audit["status"] == "FIXTURE_COMPLETE"
print("Synthetic end-to-end pipeline passed.")


Executed successfully in sequential-cell CPU fixture harness. See transcript.


### Actual-device preflight and runtime estimate
Preflight verifies all 82 author models at first/middle/last forecast positions. It checks real full-size training and inference batches, dense/sparse GPU equivalence, and exact fixture checkpoint resumption on the selected GPU.

Runtime estimates cover 11, 20 and 35 epochs; the 35-epoch estimate includes a 1.5× training margin. Setup, detector processing, export and Drive overhead are additional. No training settings are reduced if the estimate exceeds six hours.

In [5]:
# 5 — Hardware, source integrity and measured workload
OUT = BASE / "EXP016_run"
DATA = BASE / "NASA_data"
MIRROR = DRIVE_ROOT / "run"
if AUTOTEST:
    # Exercise the benchmark code on synthetic data with fixture-only dimensions.
    script = "from pathlib import Path; from exp016.io import read_json,ROOT; from exp016.train import configure; from exp016.data import audit_data; from exp016.benchmark import benchmark; c=read_json(ROOT/'config.json'); c.update(hidden=4,epochs=2,seeds=[17],training_channels=2); configure('cpu'); root=Path(" + repr(str(BASE/'fixture_data')) + "); out=Path(" + repr(str(BASE/'engineering_fixture')) + "); rows=audit_data(root,c,out); benchmark(root,rows,c,out,'cpu',steps=3)"
    worker("-c", script)
else:
    worker("-m", "exp016.run", "preflight", "--out", OUT, "--data", DATA, "--mirror", MIRROR)
    estimate = json.loads((OUT/"RUNTIME_ESTIMATE.json").read_text())
    display(estimate)


Executed successfully in sequential-cell CPU fixture harness. See transcript.


### Fit, freeze, detect and evaluate
On an interruption, reconnect to the same GPU type and run all cells again. Completed jobs are hash-verified and reused. Epoch/minibatch checkpoints preserve optimizer, early stopping, sampler and RNG state.

The Drive mirror is refreshed at checkpoints at least five minutes apart, after each completed fit, and at stage boundaries. Abrupt VM loss can require repeating work since the last durable copy. Do not edit the configuration after starting.

In [6]:
# 6 — Run or resume the full experiment
if AUTOTEST:
    worker("-m", "exp016.run", "fixture", "--out", OUT, "--data", DATA, "--mirror", MIRROR)
else:
    worker("-m", "exp016.run", "run", "--out", OUT, "--data", DATA, "--mirror", MIRROR, "--hours", "5.5")
status = json.loads((OUT/"STATUS.json").read_text())
print("RUN STATUS:", status)
if status["status"] == "PAUSED_RESUMABLE":
    print("Training conditions are unchanged. Reconnect to the same GPU type and Run all again to finish.")


Executed successfully in sequential-cell CPU fixture harness. See transcript.


### Export and inspect
A completed run includes all model states, ten-output GCN forecasts, masks, threshold receipts, metrics, logs and manifests. `RESULTS.md` is the starting point. A paused archive contains recovery state and must not be presented as a completed result.

In [7]:
# 7 — Final/partial recovery archive and download
worker("-m", "exp016.run", "export", "--out", OUT)
archive = OUT.parent / (OUT.name + "_results.zip")
temp = DRIVE_ROOT / (archive.name + ".part")
shutil.copyfile(archive,temp)
temp.replace(DRIVE_ROOT/archive.name)
print("Persistent archive:", DRIVE_ROOT/archive.name)
if (OUT/"FINAL_AUDIT.json").exists():
    print((OUT/"FINAL_AUDIT.json").read_text())
if (OUT/"RESULTS.md").exists():
    from IPython.display import Markdown, display
    display(Markdown((OUT/"RESULTS.md").read_text()))
if not AUTOTEST:
    from google.colab import files
    files.download(str(archive))
else:
    assert archive.exists()
print("Archive ready:", archive)


Executed successfully in sequential-cell CPU fixture harness. See transcript.
